# Parameters

In [38]:
show_intermediate_results = True

# Preparations

## Imports

In [ ]:
import polars as pl

playlists = pl.scan_parquet('../processed_data/data_playlist_metadata.parquet')

## Playlist Source Data

In [ ]:
playlists.collect(engine='streaming')

playlist.id,playlist.name,owner.id,owner.name,playlist.extracted_date,playlist.region,playlist.country,playlist.is_social_set,owner.is_wcs_dj,song_count,artist_count
str,str,str,str,list[str],enum,enum,bool,bool,u32,u32
"""000PP6yV32HhGWZ8NkD7Jd""","""Joensuu""","""mjannis""","""Jannis Makropulos""",[],null,null,false,false,45,42
"""000QVDaEeEmfVJpfddp611""","""Songs with Swing/Blues Shuffle…","""wcsdunedin""","""WCSMelbourne""",[],"""Oceania""","""Australia""",false,false,397,327
"""000hjS5mHvPqB6B9kD46zc""","""WSNW '25 Saturday 03:00""","""1185428002""","""Kasia Stepek""",[],"""Europe""","""Poland""",false,true,26,25
"""001F19s2D2jzePZL6kmqYM""","""SAT, 4-DEC""","""angelabehnken""","""angelabehnken""",[],null,null,false,false,12,12
"""001eKsoK3IlnY35gLuLgaC""","""🌾🌄""","""amrosso22""","""Aubrey Rosso""",[],null,null,false,false,36,17
…,…,…,…,…,…,…,…,…,…,…
"""7zxTNmWpBgXQAYpVdpqf0i""","""old pop""","""1185428002""","""Kasia Stepek""",[],"""Europe""","""Poland""",false,true,55,47
"""7zy99M6oThI8U6AqiQsoru""","""old pop""","""1185428002""","""Kasia Stepek""",[],"""Europe""","""Poland""",false,true,86,69
"""7zyfGCIIBfSpHGPMGeCP8l""","""Mel's Teaching WCS Blues Playl…","""1272924841""","""Melissa Oh Yay""",[],null,null,false,false,60,53


# Analysis

Step 1: Tokenize the playlist names by splitting on whitespaces.

We currently turn every word into its own separate keyword term.
As a later optimization, it might make sense to treat words most often
occuring together (e.g. `late night`) to make the output more useful.

In [41]:
playlists_tokenized = playlists.select(
    pl.col('playlist.id'),
    pl.col('playlist.name'),
    pl.col('playlist.name').str.to_lowercase().str.split(' ')
    .list.filter(pl.element().ne(''))
    .list.unique(maintain_order=True).alias('unique_terms'),
)

playlists_tokenized.collect(engine='streaming') if show_intermediate_results else None

playlist.id,playlist.name,unique_terms
str,str,list[str]
"""000PP6yV32HhGWZ8NkD7Jd""","""Joensuu""","[""joensuu""]"
"""000QVDaEeEmfVJpfddp611""","""Songs with Swing/Blues Shuffle…","[""songs"", ""with"", … ""timing""]"
"""000hjS5mHvPqB6B9kD46zc""","""WSNW '25 Saturday 03:00""","[""wsnw"", ""'25"", … ""03:00""]"
"""001F19s2D2jzePZL6kmqYM""","""SAT, 4-DEC""","[""sat,"", ""4-dec""]"
"""001eKsoK3IlnY35gLuLgaC""","""🌾🌄""","[""🌾🌄""]"
…,…,…
"""7zxTNmWpBgXQAYpVdpqf0i""","""old pop""","[""old"", ""pop""]"
"""7zy99M6oThI8U6AqiQsoru""","""old pop""","[""old"", ""pop""]"
"""7zyfGCIIBfSpHGPMGeCP8l""","""Mel's Teaching WCS Blues Playl…","[""mel's"", ""teaching"", … ""playlist""]"


Step 2: Aggregate over playlist terms

In [ ]:
exploded_playlists_tokenized = playlists_tokenized\
    .explode('unique_terms')\
    .rename({'unique_terms': 'term'})

exploded_playlists_tokenized.limit(100).collect(engine='streaming') if show_intermediate_results else None

playlist.id,playlist.name,term
str,str,str
"""000PP6yV32HhGWZ8NkD7Jd""","""Joensuu""","""joensuu"""
"""000QVDaEeEmfVJpfddp611""","""Songs with Swing/Blues Shuffle…","""songs"""
"""000QVDaEeEmfVJpfddp611""","""Songs with Swing/Blues Shuffle…","""with"""
"""000QVDaEeEmfVJpfddp611""","""Songs with Swing/Blues Shuffle…","""swing/blues"""
"""000QVDaEeEmfVJpfddp611""","""Songs with Swing/Blues Shuffle…","""shuffle"""
…,…,…
"""00BrxSphwuS0d6xb4iI0b5""","""Best WCS by BPM Old""","""wcs"""
"""00BrxSphwuS0d6xb4iI0b5""","""Best WCS by BPM Old""","""by"""
"""00BrxSphwuS0d6xb4iI0b5""","""Best WCS by BPM Old""","""bpm"""


In [34]:
tokens = exploded_playlists_tokenized\
    .group_by('term')\
    .agg(pl.col('term').count().alias('playlist_count'))\
    .sort('playlist_count', descending=True)

Review query plan for potential performance/memory problems:

In [ ]:
tokens.show_graph(plan_stage='physical', engine='streaming', optimized=True)

In [46]:
tokens.filter(pl.col('playlist_count').ge(100)).collect(engine='streaming')

term,playlist_count
str,u32
"""wcs""",14539
"""-""",6276
"""swing""",2652
"""blues""",1961
"""bpm""",1957
…,…
"""finals""",102
"""mar""",102
"""cool""",102
